# STEP 1. 적재 + 스키마 통일 + 범위 필터 + 완전성 검증

## 이번 개편에서 바뀐 것

| | 이전 | 이번 |
|---|---|---|
| 점포 총수 | 2023·24는 `점포_수`(=비프랜차이즈), 2025~는 `전체_점포_수` → **정의 불연속** | 양쪽 모두 `전체_점포_수`(총수)로 통일 |
| `프랜차이즈_점포_수` | 미사용 | 보존 (프랜차이즈 비율 파생 가능) |
| 필터 시점 | 100개 업종을 STEP 5까지 끌고 감 | **STEP 1에서 10개 업종 · 12분기로 축소** |
| 완전성 검증 | 없음 | 교차표 + 분기 연속성 진단 추가 |
| 원본 | 덮어씀 | `_raw` 보존 후 `.copy()` 로 작업 |

## 점포_수 매핑을 왜 바꿨나

원본 파일 두 세대에서 다음 항등식이 **100%** 성립합니다.

```
2023·2024 :  유사_업종_점포_수 = 점포_수     + 프랜차이즈_점포_수
2025~     :  전체_점포_수     = 일반_점포_수 + 프랜차이즈_점포_수
```

즉 구파일의 `점포_수`는 총수가 아니라 **비프랜차이즈 점포수**입니다. 이전 코드는 구파일의 `점포_수`(부분)와 신파일의 `전체_점포_수`(총수)를 같은 컬럼에 넣어서, 2024Q4 → 2025Q1 사이에 평균이 **인위적으로 +18% 점프**했습니다.

과소 정도는 업종마다 다릅니다 — 치킨 64%, 패스트푸드 50%, 한식 10%. 처리군은 거의 안 깎이고 프랜차이즈 업종만 크게 깎여서 **매칭 공변량 자체가 왜곡**되어 있었습니다.

## 분석 범위

- **12분기** 20231 ~ 20261
- **10개 외식업**, 코드(`CS100001`~`CS100010`)로 지정

> ⚠️ 라벨 `y(t)`는 `t+1` 분기를 봅니다. 20261을 쓰지 않으므로 **라벨이 붙는 구간은 20231~20261 (12분기)** 입니다. 20254 행은 STEP 3에서 라벨 없음으로 탈락합니다.


In [1]:
import sys, os
from pathlib import Path

# 노트북이 어디서 열리든 프로젝트 루트를 찾아 sys.path에 추가
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
print("프로젝트 루트:", ROOT)

프로젝트 루트: c:\Users\spide\ai-data-bootcamp\project\h2_nb


In [2]:
from config import (RAW, PROC, ENCODING, FILES, QUARTERS, SVC_CODES, FOOD,
                    STORE_RENAME_OLD, STORE_KEEP, SALES_KEEP, TARGET_TYPE)

CODES = list(SVC_CODES.keys())
print(f"분석 분기 {len(QUARTERS)}개 : {QUARTERS[0]} ~ {QUARTERS[-1]}")
print(f"분석 업종 {len(CODES)}개 : {', '.join(FOOD)}")

def load(key):
    p = RAW / FILES[key]
    df = pd.read_csv(p, encoding=ENCODING, low_memory=False)
    print(f"  {key:12} {df.shape[0]:>8,}행 × {df.shape[1]:>2}열   {p.name}")
    return df

분석 분기 13개 : 20231 ~ 20261
분석 업종 10개 : 한식음식점, 중식음식점, 일식음식점, 양식음식점, 제과점, 패스트푸드점, 치킨전문점, 분식전문점, 호프-간이주점, 커피-음료


## 1-1. 원본 적재 (`_raw` 보존)

`_raw` 는 절대 수정하지 않습니다. 중간에 뭔가 잘못되면 이걸로 되돌아옵니다.

In [3]:
print("[점포]")
store23_raw = load("store_2023")
store24_raw = load("store_2024")
store25_raw = load("store_2025")

print("[매출]")
sales23_raw   = load("sales_2023")
sales24_raw   = load("sales_2024")
sales25_raw   = load("sales_2025")
sales_late_raw = load("sales_late")

print("[상권 속성]")
area_raw     = load("area")
flow_raw     = load("flow")
facility_raw = load("facility")

[점포]
  store_2023    307,741행 × 14열   서울시_상권분석서비스(점포-상권)_2023년.csv
  store_2024    306,889행 × 14열   서울시 상권분석서비스(점포-상권)_2024년.csv
  store_2025    380,747행 × 14열   서울시 상권분석서비스(점포-상권).csv
[매출]
  sales_2023     88,246행 × 55열   서울시_상권분석서비스(추정매출-상권)_2023년.csv
  sales_2024     87,179행 × 55열   서울시 상권분석서비스(추정매출-상권)_2024년.csv
  sales_2025     85,732행 × 55열   서울시 상권분석서비스(추정매출-상권)_2025년.csv
  sales_late    106,920행 × 55열   서울시 상권분석서비스(추정매출-상권).csv
[상권 속성]
  area            1,650행 × 11열   서울시 상권분석서비스(영역-상권).csv
  flow           34,633행 × 27열   서울시 상권분석서비스(길단위인구-상권).csv
  facility       33,138행 × 25열   서울시 상권분석서비스(집객시설-상권).csv


## 1-2. 점포 스키마 진단

리네임 **전에** 무엇이 다른지 눈으로 확인합니다.

In [4]:
c23, c24, c25 = (set(d.columns) for d in (store23_raw, store24_raw, store25_raw))
print("2023 vs 2024      :", c23 ^ c24 or "차이 없음")
print("2024 vs 2025~     :", c24 ^ c25)
print()
print("[항등식 검증] 두 세대가 같은 구조인지 산술로 확인")
for nm, d, tot, gen in [("2023 ", store23_raw, "유사_업종_점포_수", "점포_수"),
                        ("2024 ", store24_raw, "유사_업종_점포_수", "점포_수"),
                        ("2025~", store25_raw, "전체_점포_수",     "일반_점포_수")]:
    hit = (d[tot] == d[gen] + d["프랜차이즈_점포_수"]).mean()
    print(f"  {nm}  {tot:14} == {gen:12} + 프랜차이즈_점포_수   일치율 {hit:.4f}")
print()
print("→ 구파일의 '점포_수'는 총수가 아니라 비프랜차이즈. '유사_업종_점포_수'가 총수다.")

2023 vs 2024      : 차이 없음
2024 vs 2025~     : {'일반_점포_수', '유사_업종_점포_수', '점포_수', '전체_점포_수'}

[항등식 검증] 두 세대가 같은 구조인지 산술로 확인
  2023   유사_업종_점포_수     == 점포_수         + 프랜차이즈_점포_수   일치율 1.0000
  2024   유사_업종_점포_수     == 점포_수         + 프랜차이즈_점포_수   일치율 1.0000
  2025~  전체_점포_수        == 일반_점포_수      + 프랜차이즈_점포_수   일치율 1.0000

→ 구파일의 '점포_수'는 총수가 아니라 비프랜차이즈. '유사_업종_점포_수'가 총수다.


## 1-3. 스키마 통일 — 신파일 이름 기준

`.copy()` 로 작업본을 만들고 구파일 컬럼명을 신파일에 맞춥니다.

In [5]:
store23 = store23_raw.copy().rename(columns=STORE_RENAME_OLD)
store24 = store24_raw.copy().rename(columns=STORE_RENAME_OLD)
store25 = store25_raw.copy()

c23, c24, c25 = (set(d.columns) for d in (store23, store24, store25))
assert not (c23 ^ c24) and not (c24 ^ c25), "리네임 후에도 컬럼이 어긋납니다"
print("리네임 후 컬럼 차이 : 없음 ✅")

store = pd.concat([store23, store24, store25], axis=0, ignore_index=True)
assert len(store) == len(store23) + len(store24) + len(store25), "concat 중 행 손실"
print(f"병합 {len(store):,}행  (= {len(store23):,} + {len(store24):,} + {len(store25):,}) ✅")
print(f"중복 행 {store.duplicated().sum():,}개")
print(f"원본 분기 {sorted(store['기준_년분기_코드'].unique())}")
print("  → 20261 은 다음 셀에서 잘려나갑니다 (분석 범위 12분기)")

리네임 후 컬럼 차이 : 없음 ✅
병합 995,377행  (= 307,741 + 306,889 + 380,747) ✅
중복 행 0개
원본 분기 [np.int64(20231), np.int64(20232), np.int64(20233), np.int64(20234), np.int64(20241), np.int64(20242), np.int64(20243), np.int64(20244), np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254), np.int64(20261)]
  → 20261 은 다음 셀에서 잘려나갑니다 (분석 범위 12분기)


### 정의 연속성 검증

이번 수정이 실제로 불연속을 없앴는지 확인합니다. 골목상권 외식업 분기별 평균 점포수입니다.

`전체_점포_수` 는 2024Q4 → 2025Q1 에서 매끄럽게 이어져야 하고, `일반_점포_수` 도 마찬가지여야 합니다. 이전 코드가 만들던 값(구파일은 일반, 신파일은 전체)은 여기서 점프가 보입니다.

In [6]:
chk = store[(store["상권_구분_코드_명"] == TARGET_TYPE) &
            (store["서비스_업종_코드"].isin(CODES))]
t = chk.groupby("기준_년분기_코드")[["전체_점포_수", "일반_점포_수"]].mean()
t["이전코드가_넣던_값"] = np.where(t.index >= 20251, t["전체_점포_수"], t["일반_점포_수"])
display(t.round(2))
print("※ '전체_점포_수'는 20244(6.23) → 20251(6.16) 로 연속.")
print("   '이전코드가_넣던_값'은 5.22 → 6.16 으로 +18% 점프 — 이게 고친 문제입니다.")
print("   (20261 행은 참고용. 분석에는 쓰지 않습니다.)")

,전체_점포_수,일반_점포_수,이전코드가_넣던_값
기준_년분기_코드,,,
20231,6.23,5.21,5.21
20232,6.24,5.22,5.22
20233,6.24,5.22,5.22
20234,6.26,5.23,5.23
20241,6.29,5.26,5.26
20242,6.29,5.27,5.27
20243,6.25,5.24,5.24
20244,6.23,5.22,5.22
20251,6.16,5.17,6.16


※ '전체_점포_수'는 20244(6.23) → 20251(6.16) 로 연속.
   '이전코드가_넣던_값'은 5.22 → 6.16 으로 +18% 점프 — 이게 고친 문제입니다.
   (20261 행은 참고용. 분석에는 쓰지 않습니다.)


## 1-4. 상권 단위 분모 계산 — **업종 필터 전에**

업종을 10개로 줄이고 나면 `상권_전체점포`(100개 업종 합)를 만들 수 없습니다. 분모는 반드시 지금 계산해야 합니다.

In [7]:
store = store[store["기준_년분기_코드"].isin(QUARTERS)].copy()
print(f"12분기 필터 후 {len(store):,}행 (업종 {store['서비스_업종_코드'].nunique()}개)")

key = ["기준_년분기_코드", "상권_코드"]
전체 = store.groupby(key)["전체_점포_수"].sum().rename("상권_전체점포")
외식 = (store[store["서비스_업종_코드"].isin(CODES)]
        .groupby(key)["전체_점포_수"].sum().rename("상권_외식점포"))

denom = pd.concat([전체, 외식], axis=1).fillna({"상권_외식점포": 0}).reset_index()
denom["외식_비중"] = denom["상권_외식점포"] / denom["상권_전체점포"].replace(0, np.nan)
print(f"분모 테이블 {denom.shape}")
display(denom[["상권_전체점포", "상권_외식점포", "외식_비중"]].describe().round(3))

12분기 필터 후 995,377행 (업종 100개)
분모 테이블 (21450, 5)


,상권_전체점포,상권_외식점포,외식_비중
count,21450.000,21450.000,21450.000
mean,319.382,81.098,0.265
std,673.082,140.279,0.116
min,1.000,0.000,0.000
25%,72.000,16.000,0.193
50%,155.000,39.000,0.257
75%,297.000,84.000,0.328
max,15131.000,1724.000,0.747


## 1-5. 업종·컬럼 필터

여기서 데이터가 1/10로 줄어듭니다. 이후 모든 단계가 가벼워집니다.

In [8]:
before = len(store)
store = store[store["서비스_업종_코드"].isin(CODES)][STORE_KEEP].copy()
store = store.merge(denom, on=key, how="left", validate="many_to_one")

store["프랜차이즈_비율"] = store["프랜차이즈_점포_수"] / store["전체_점포_수"].replace(0, np.nan)

print(f"업종 필터 {before:,} → {len(store):,}행  ({len(store)/before:.1%})")
print(f"컬럼 {len(store.columns)}개")
display(store.head(3))

업종 필터 995,377 → 160,352행  (16.1%)
컬럼 15개


,기준_년분기_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,전체_점포_수,일반_점포_수,프랜차이즈_점포_수,개업_점포_수,폐업_점포_수,상권_전체점포,상권_외식점포,외식_비중,프랜차이즈_비율
0,20231,골목상권,3110001,이북5도청사,CS100001,한식음식점,11,10,1,1,0,49,19.0,0.387755,0.090909
1,20231,골목상권,3110001,이북5도청사,CS100003,일식음식점,1,1,0,0,0,49,19.0,0.387755,0.000000
2,20231,골목상권,3110001,이북5도청사,CS100008,분식전문점,3,3,0,0,0,49,19.0,0.387755,0.000000


## 1-6. 완전성 검증 ①  교차표

자치구 단위 데이터라면 이 표가 25(구 개수)로 꽉 찹니다. **상권 단위는 다릅니다** — 업종마다 존재하는 상권 수가 다르고, 분기마다 흔들립니다. 그 흔들림이 STEP 3에서 라벨을 못 만드는 칸의 정체입니다.

In [9]:
gm = store[store["상권_구분_코드_명"] == TARGET_TYPE]
ct = pd.crosstab(gm["기준_년분기_코드"], gm["서비스_업종_코드_명"], margins=True)
ct = ct[[c for c in FOOD if c in ct.columns] + ["All"]]
display(ct)

n_area = gm["상권_코드"].nunique()
print(f"{TARGET_TYPE} 상권 {n_area}개 × 13분기 = {n_area*13:,} 이 업종별 만석 기준")
cov = (gm.groupby("서비스_업종_코드_명").size() / (n_area * 13)).sort_values(ascending=False)
display(cov.mul(100).round(1).rename("커버리지(%)").to_frame())
print("※ 커버리지가 낮은 업종일수록 대조군으로 쓸 수 있는 상권 풀이 좁다.")

서비스_업종_코드_명,한식음식점,중식음식점,일식음식점,양식음식점,제과점,패스트푸드점,치킨전문점,분식전문점,호프-간이주점,커피-음료,All
기준_년분기_코드,,,,,,,,,,,
20231,1047,708,609,663,744,721,746,855,850,1015,7958
20232,1043,710,611,666,748,718,747,849,849,1019,7960
20233,1042,710,608,659,749,717,742,850,850,1019,7946
20234,1040,705,607,656,752,718,747,851,855,1023,7954
20241,1041,703,614,658,754,716,747,858,853,1024,7968
20242,1042,697,609,661,753,705,740,859,850,1023,7939
20243,1043,693,604,656,749,699,735,859,852,1025,7915
20244,1044,693,605,647,751,693,733,858,851,1027,7902
20251,1044,690,601,651,753,692,726,856,848,1025,7886


골목상권 상권 1084개 × 13분기 = 14,092 이 업종별 만석 기준


,커버리지(%)
서비스_업종_코드_명,
한식음식점,96.2
커피-음료,94.3
분식전문점,78.7
호프-간이주점,78.3
제과점,69.4
치킨전문점,67.7
패스트푸드점,64.6
중식음식점,64.2
양식음식점,60.3


※ 커버리지가 낮은 업종일수록 대조군으로 쓸 수 있는 상권 풀이 좁다.


## 1-7. 완전성 검증 ②  분기 연속성

`(상권, 업종)` 조합이 12분기를 모두 갖고 있는지 봅니다. **12개 미만인 칸은 중간에 분기가 끊겨서 STEP 3에서 라벨이 일부 만들어지지 않습니다.**

In [10]:
cnt = (store.groupby(["상권_코드", "서비스_업종_코드"])["기준_년분기_코드"]
       .nunique().rename("관측분기수"))
dist = cnt.value_counts().sort_index(ascending=False)
display(pd.DataFrame({"칸수": dist, "비율": (dist/len(cnt)).round(3)}))
print(f"전체 (상권×업종) 조합 {len(cnt):,}개 중 12분기 완비 {(cnt==12).sum():,}개 ({(cnt==12).mean():.1%})")

# 연속성: 관측된 분기가 중간에 비지 않고 이어지는가
qidx = {q: i for i, q in enumerate(QUARTERS)}
def is_gapless(s):
    v = sorted(qidx[q] for q in s)
    return v == list(range(v[0], v[0] + len(v)))
gap = store.groupby(["상권_코드", "서비스_업종_코드"])["기준_년분기_코드"].apply(is_gapless)
print(f"관측 분기가 중간에 비어 있는 칸 : {(~gap).sum():,}개 ({(~gap).mean():.1%})")
print("→ STEP 3 의 '분기 연속성' 검사가 걸러낼 대상")

,칸수,비율
관측분기수,,
13,11522,0.881
12,141,0.011
11,147,0.011
10,150,0.011
9,140,0.011
8,120,0.009
7,126,0.010
6,131,0.010
5,142,0.011


전체 (상권×업종) 조합 13,074개 중 12분기 완비 141개 (1.1%)
관측 분기가 중간에 비어 있는 칸 : 139개 (1.1%)
→ STEP 3 의 '분기 연속성' 검사가 걸러낼 대상


## 1-8. 매출 — 중복 제거 후 필터

`추정매출-상권_2025년.csv` 와 연도 표기 없는 최신 파일이 20251~20254 구간에서 겹칩니다. 키 기준으로 최신 파일 값을 남깁니다.

In [11]:
sales_parts = []
for nm, raw in [("2023", sales23_raw), ("2024", sales24_raw),
                ("2025", sales25_raw), ("late", sales_late_raw)]:
    d = raw.copy()
    d = d[d["기준_년분기_코드"].isin(QUARTERS) & d["서비스_업종_코드"].isin(CODES)][SALES_KEEP]
    print(f"  {nm:5} {len(d):>7,}행  분기 {sorted(d['기준_년분기_코드'].unique())}")
    sales_parts.append(d)

sales = pd.concat(sales_parts, ignore_index=True)
skey = ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"]
before = len(sales)
sales = sales.drop_duplicates(subset=skey, keep="last").reset_index(drop=True)
print(f"\n중복 제거 {before:,} → {len(sales):,}  (-{before-len(sales):,})")
assert not sales.duplicated(skey).any(), "키 중복 잔존"
print(f"분기 커버리지 {sorted(sales['기준_년분기_코드'].unique())}")

  2023   27,430행  분기 [np.int64(20231), np.int64(20232), np.int64(20233), np.int64(20234)]
  2024   27,095행  분기 [np.int64(20241), np.int64(20242), np.int64(20243), np.int64(20244)]
  2025   26,460행  분기 [np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254)]
  late   33,033행  분기 [np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254), np.int64(20261)]

중복 제거 114,018 → 87,558  (-26,460)
분기 커버리지 [np.int64(20231), np.int64(20232), np.int64(20233), np.int64(20234), np.int64(20241), np.int64(20242), np.int64(20243), np.int64(20244), np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254), np.int64(20261)]


## 1-9. 상권 속성 3종

`영역-상권` 은 시간에 안 변하므로 분기 필터가 없습니다. `길단위인구`·`집객시설` 은 2021년부터 있으므로 12분기로 자릅니다.

이 셋은 행 수가 작아서(수만 행) 컬럼을 다 남깁니다 — 보조질문 2에서 성별·연령대·시설 종류별 변수를 쓸 여지를 남겨둡니다.

In [12]:
AREA_KEEP = ["상권_코드", "상권_구분_코드_명", "상권_코드_명", "자치구_코드", "자치구_코드_명",
             "행정동_코드", "행정동_코드_명", "영역_면적", "엑스좌표_값", "와이좌표_값"]

area = area_raw.copy()[AREA_KEEP]
assert area["상권_코드"].is_unique, "영역-상권에 상권_코드 중복"
print(f"area     {area.shape}  상권 {area['상권_코드'].nunique():,}개")
display(area["상권_구분_코드_명"].value_counts().to_frame("상권 수"))

flow = flow_raw.copy()
flow = flow[flow["기준_년분기_코드"].isin(QUARTERS)].drop(
    columns=["상권_구분_코드", "상권_구분_코드_명", "상권_코드_명"])
print(f"flow     {flow.shape}  키중복 {flow.duplicated(['기준_년분기_코드','상권_코드']).sum()}")

facility = facility_raw.copy()
facility = facility[facility["기준_년분기_코드"].isin(QUARTERS)].drop(
    columns=["상권_구분_코드", "상권_구분_코드_명", "상권_코드_명"])
print(f"facility {facility.shape}  키중복 {facility.duplicated(['기준_년분기_코드','상권_코드']).sum()}")
print(f"         집객시설 커버 상권 {facility['상권_코드'].nunique():,}개 "
      f"(영역-상권 {area['상권_코드'].nunique():,}개 중)")

area     (1650, 10)  상권 1,650개


,상권 수
상권_구분_코드_명,
골목상권,1090
전통시장,305
발달상권,249
관광특구,6


flow     (21434, 24)  키중복 0
facility (20514, 22)  키중복 0
         집객시설 커버 상권 1,578개 (영역-상권 1,650개 중)


## 1-10. 저장

이전 대비 파일 크기가 크게 줄어듭니다 (점포 166MB → 수 MB).

In [13]:
out = {"01_store.pkl": store, "01_sales.pkl": sales,
       "01_area.pkl": area, "01_flow.pkl": flow, "01_facility.pkl": facility}
for nm, d in out.items():
    d.to_pickle(PROC / nm)
    mb = (PROC / nm).stat().st_size / 1024**2
    print(f"[저장] {nm:16} {str(d.shape):>16}   {mb:6.1f} MB")

print("\n다음 단계 STEP 2 는 아래를 그대로 씁니다.")
print("  점포 총수 컬럼 = '전체_점포_수'  (이전의 '점포_수' 아님)")
print("  상권_전체점포 / 상권_외식점포 / 외식_비중 / 프랜차이즈_비율 은 STEP 1 에서 이미 계산됨")

[저장] 01_store.pkl         (160352, 15)     26.8 MB
[저장] 01_sales.pkl           (87558, 5)      4.0 MB
[저장] 01_area.pkl            (1650, 10)      0.2 MB
[저장] 01_flow.pkl           (21434, 24)      3.9 MB
[저장] 01_facility.pkl       (20514, 22)      3.4 MB

다음 단계 STEP 2 는 아래를 그대로 씁니다.
  점포 총수 컬럼 = '전체_점포_수'  (이전의 '점포_수' 아님)
  상권_전체점포 / 상권_외식점포 / 외식_비중 / 프랜차이즈_비율 은 STEP 1 에서 이미 계산됨


---

## STEP 2 에서 손봐야 할 것

- 점포 총수 컬럼명이 `점포_수` → **`전체_점포_수`** 로 바뀌었습니다
- `상권_전체점포`, `상권_외식점포` 를 STEP 2 에서 다시 만들면 안 됩니다 (여기서 만들었음)
- 좌결합 후 행 수 검증 로직은 그대로 유지
